In [2]:
from collections import defaultdict
import numpy as np
import pandas as pd

from routellm.evals.mmlu.domains import ALL_MMLU_DOMAINS

routers = ["mf", "causal_llm", "sw_ranking", "bert"]

domain_cache = {}
for domain in ALL_MMLU_DOMAINS:
	domain_cache[domain] = np.load(f"routellm/evals/mmlu/cache_{domain}.npy", allow_pickle=True).item()

for router in routers:
	all_router = pd.concat([domain_cache[domain][router] for domain in ALL_MMLU_DOMAINS])
	print("router median across full MMLU", router, all_router.min(), all_router.max())

router median across full MMLU mf 0.05041242763400078 0.5883079767227173
router median across full MMLU causal_llm 0.045430123805999756 0.7121628522872925
router median across full MMLU sw_ranking 0.2126316839892699 0.23674457227520596
router median across full MMLU bert 0.10695987939834595 0.9343840852379799


In [3]:
for router in routers:
	all_router = pd.concat([domain_cache[domain][router] for domain in ALL_MMLU_DOMAINS])
	for percentile in [0.2, 0.5, 0.8]:
		threshold = all_router.quantile(percentile)
		for domain in ["marketing", "college_mathematics", "philosophy"]:
			print(router, percentile, domain, "threshold", threshold, "count", (domain_cache[domain][router] >= threshold).sum() / len(domain_cache[domain][router]))

mf 0.2 marketing threshold 0.178583499789238 count 0.49145299145299143
mf 0.2 college_mathematics threshold 0.178583499789238 count 1.0
mf 0.2 philosophy threshold 0.178583499789238 count 0.752411575562701
mf 0.5 marketing threshold 0.24183201789855957 count 0.1282051282051282
mf 0.5 college_mathematics threshold 0.24183201789855957 count 0.98
mf 0.5 philosophy threshold 0.24183201789855957 count 0.3504823151125402
mf 0.8 marketing threshold 0.31404213309288026 count 0.021367521367521368
mf 0.8 college_mathematics threshold 0.31404213309288026 count 0.75
mf 0.8 philosophy threshold 0.31404213309288026 count 0.08038585209003216
causal_llm 0.2 marketing threshold 0.10563856363296509 count 0.3333333333333333
causal_llm 0.2 college_mathematics threshold 0.10563856363296509 count 1.0
causal_llm 0.2 philosophy threshold 0.10563856363296509 count 0.8938906752411575
causal_llm 0.5 marketing threshold 0.148226797580719 count 0.05555555555555555
causal_llm 0.5 college_mathematics threshold 0.148

In [34]:
res = []
router_values = defaultdict(list)
for router in ["mf", "causal_llm", "sw_ranking", "bert"]:
	for domain in domain_cache:
		res.append({
			"router": router,
			"domain": domain,
			"mean": domain_cache[domain][router].mean()
		})

res = pd.DataFrame(res)

for router in ["mf", "causal_llm", "sw_ranking", "bert"]:
	print(router)
	print("=" * 50)
	print((res[res["router"] == router].sort_values("mean", ascending=True))[["domain", "mean"]].to_string(index=False))

llm_marketing = domain_cache["elementary_mathematics"]["causal_llm"]
print(llm_marketing[llm_marketing <0.169700].count() / len(llm_marketing))

mf
                             domain     mean
                   security_studies 0.169700
             high_school_us_history 0.171070
                          sociology 0.182916
                          marketing 0.183776
       high_school_european_history 0.183945
                   public_relations 0.190677
                  us_foreign_policy 0.190793
                        human_aging 0.195618
                  logical_fallacies 0.196499
          high_school_world_history 0.198994
                     moral_disputes 0.199654
            professional_psychology 0.204363
high_school_government_and_politics 0.206615
                         management 0.207283
                          nutrition 0.207625
                    business_ethics 0.209393
                           virology 0.209699
             high_school_psychology 0.211935
                      jurisprudence 0.217412
                    human_sexuality 0.218101
              high_school_geography 0.218330
       

In [30]:
per_domain = []
for domain in domain_cache:
	per_domain.append({
		"domain": domain,
		"mean": res[res["domain"] == domain][["mean"]].mean().item()
	})
# print(per_domain)
df = pd.DataFrame(per_domain)
print(df.sort_values("mean", ascending=True).to_string(index=False))

                             domain     mean
                          marketing 0.218432
                         management 0.225950
                   public_relations 0.233684
                   security_studies 0.234455
                        human_aging 0.238473
                          sociology 0.239086
            professional_psychology 0.241990
high_school_government_and_politics 0.246062
              professional_medicine 0.246564
             high_school_us_history 0.246691
                          nutrition 0.252700
                           virology 0.252817
             high_school_psychology 0.254321
                  us_foreign_policy 0.255914
       high_school_european_history 0.258579
                  international_law 0.260459
                 clinical_knowledge 0.263044
                    business_ethics 0.263501
                      jurisprudence 0.264124
                    human_sexuality 0.264852
                     moral_disputes 0.265021
          

In [38]:
from tqdm import tqdm

domains = ALL_MMLU_DOMAINS

contaminated_prompts = pd.read_json(
	f"routellm/evals/mmlu/contaminated_prompts.jsonl", lines=True
)["eval_prompt"].tolist()

contaminated_gsm8k = pd.read_json(
	f"routellm/evals/gsm8k/contaminated_prompts.jsonl", lines=True
)["eval_prompt"].tolist()

for router in routers:
	print(router)
	# print("=" * 50)
	# all_data = pd.DataFrame()
	# for domain in tqdm(domains, desc="Loading domain data"):
	# 	data = pd.read_csv(f"routellm/evals/mmlu/responses/mmlu_{domain}.csv")
	# 	data = data[~data["prompt"].isin(contaminated_prompts)]
	# 	cache = domain_cache[domain][router]
	# 	assert len(data) == len(cache)
	# 	data["value"] = cache
	# 	all_data = pd.concat([all_data, data], ignore_index=True)
	all_data = pd.read_csv(f"routellm/evals/gsm8k/gsm8k_responses.csv")
	all_data = all_data[~all_data["prompt"].isin(contaminated_gsm8k)]
	cache = np.load("routellm/evals/gsm8k/cache.npy", allow_pickle=True).item()
	print(cache)
	assert len(cache[router]) == len(all_data), (len(cache), len(all_data))

	print(len(all_data))

	strong = "gpt-4-1106-preview"
	weak = "mistralai/Mixtral-8x7B-Instruct-v0.1"
	both_true = all_data[(all_data[weak] == True) & (all_data[strong] == True)]
	both_false = all_data[(all_data[weak] == False) & (all_data[strong] == False)]
	both_same = pd.concat([both_true, both_false], ignore_index=True)
	different = all_data[all_data[weak] != all_data[strong]]

	hard = all_data[(all_data[strong] == True) & (all_data[weak] == False)]
	easy = all_data[(all_data[strong] == False) & (all_data[weak] == True)]
	print("hard", hard["value"].mean())
	print("all", all_data["value"].mean())
	print("normalized_diference", (hard["value"].mean() - all_data["value"].mean()) / all_data["value"].std())

	print("both_same", both_same["value"].mean())
	print("different", different["value"].mean())
	print("both true", both_true["value"].mean())
	print("both false", both_false["value"].mean())

mf
{'random': 3       0.944899
4       0.802312
5       0.041081
6       0.324944
7       0.327917
          ...   
1314    0.818300
1315    0.470424
1316    0.437882
1317    0.863634
1318    0.192669
Name: prompt, Length: 1307, dtype: float64}


KeyError: 'mf'